<a href="https://colab.research.google.com/github/EmePin/Analisis-de-datos/blob/main/Fake_News_2_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os
!pip install tensorflow

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

import gradio as gr
import joblib


In [2]:
# Descargar dataset
path = kagglehub.dataset_download("subho117/fake-news-detection-using-machine-learning")

df_full = pd.read_csv(os.path.join(path, "News.csv")) # Cargar el CSV completo primero

# Imprimir las columnas disponibles para verificar y que el usuario sepa
print("Columnas disponibles en el dataset original:", df_full.columns.tolist())

# Asumiendo que la columna de contenido se llama 'text'.
# Si es diferente, por favor modifica 'content_column_name' a la columna correcta.
content_column_name = 'text' # Columna asumida para el contenido

if content_column_name in df_full.columns:
    # Seleccionamos 'title', la columna de contenido y 'class', y luego eliminamos filas con NaNs
    df = df_full[['title', content_column_name, 'class']].dropna()
    df['full_text'] = df['title'].astype(str) + ' ' + df[content_column_name].astype(str)
    X = df['full_text'].values
    print(f"La variable X ahora combina 'title' y '{content_column_name}'.")
else:
    print(f"Advertencia: No se encontró la columna '{content_column_name}'. Usando solo 'title' para X.")
    df = df_full[['title', 'class']].dropna()
    X = df['title'].astype(str).values

y = df['class'].astype(int).values

print("Las primeras 3 entradas de X (después de la posible concatenación):")
print(X[:3])

Using Colab cache for faster access to the 'fake-news-detection-using-machine-learning' dataset.
Columnas disponibles en el dataset original: ['Unnamed: 0', 'title', 'text', 'subject', 'date', 'class']
La variable X ahora combina 'title' y 'text'.
Las primeras 3 entradas de X (después de la posible concatenación):
[' Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my frie

In [3]:
print(df)

                                                   title  \
0       Donald Trump Sends Out Embarrassing New Year’...   
1       Drunk Bragging Trump Staffer Started Russian ...   
2       Sheriff David Clarke Becomes An Internet Joke...   
3       Trump Is So Obsessed He Even Has Obama’s Name...   
4       Pope Francis Just Called Out Donald Trump Dur...   
...                                                  ...   
44914  'Fully committed' NATO backs new U.S. approach...   
44915  LexisNexis withdrew two products from Chinese ...   
44916  Minsk cultural hub becomes haven from authorities   
44917  Vatican upbeat on possibility of Pope Francis ...   
44918  Indonesia to buy $1.14 billion worth of Russia...   

                                                    text  class  \
0      Donald Trump just couldn t wish all Americans ...      0   
1      House Intelligence Committee Chairman Devin Nu...      0   
2      On Friday, it was revealed that former Milwauk...      0   
3      On C

In [4]:
print(df)

                                                   title  \
0       Donald Trump Sends Out Embarrassing New Year’...   
1       Drunk Bragging Trump Staffer Started Russian ...   
2       Sheriff David Clarke Becomes An Internet Joke...   
3       Trump Is So Obsessed He Even Has Obama’s Name...   
4       Pope Francis Just Called Out Donald Trump Dur...   
...                                                  ...   
44914  'Fully committed' NATO backs new U.S. approach...   
44915  LexisNexis withdrew two products from Chinese ...   
44916  Minsk cultural hub becomes haven from authorities   
44917  Vatican upbeat on possibility of Pope Francis ...   
44918  Indonesia to buy $1.14 billion worth of Russia...   

                                                    text  class  \
0      Donald Trump just couldn t wish all Americans ...      0   
1      House Intelligence Committee Chairman Devin Nu...      0   
2      On Friday, it was revealed that former Milwauk...      0   
3      On C

In [5]:
print(df.head())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text  class  \
0  Donald Trump just couldn t wish all Americans ...      0   
1  House Intelligence Committee Chairman Devin Nu...      0   
2  On Friday, it was revealed that former Milwauk...      0   
3  On Christmas day, Donald Trump announced that ...      0   
4  Pope Francis used his annual Christmas Day mes...      0   

                                           full_text  
0   Donald Trump Sends Out Embarrassing New Year’...  
1   Drunk Bragging Trump Staffer Started Russian ...  
2   Sheriff David Clarke Becomes An Internet Joke...  
3   Trump Is So Obsessed He Even Has Obama’s Name...  
4   Pope 

In [6]:
VOCAB_SIZE = 10000 # Aumentado debido a la inclusión del contenido completo
MAX_LEN = 500 # Aumentado para acomodar el contenido completo de las noticias

tokenizer = Tokenizer(num_words=VOCAB_SIZE)# convierte palabras → números
tokenizer.fit_on_texts(X) # Diccionario

X_seq = tokenizer.texts_to_sequences(X)
# Convierte cada noticia en una lista de números
# Ejemplo: "economy grows fast" → [45, 102, 78]
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN)
# Ajusta todas las noticias al mismo tamaño
# Si son cortas → agrega ceros (padding)
# Si son largas → las recorta

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42, stratify=y
)


In [8]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)


Epoch 1/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 554s 1s/step - accuracy: 0.9489 - loss: 0.1447 - val_accuracy: 0.9840 - val_loss: 0.0590
Epoch 2/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 553s 1s/step - accuracy: 0.9837 - loss: 0.0532 - val_accuracy: 0.9875 - val_loss: 0.0449
Epoch 3/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 555s 1s/step - accuracy: 0.9887 - loss: 0.0406 - val_accuracy: 0.9505 - val_loss: 0.1716
Epoch 4/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 561s 1s/step - accuracy: 0.9794 - loss: 0.0623 - val_accuracy: 0.9894 - val_loss: 0.0370
Epoch 5/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 556s 1s/step - accuracy: 0.9757 - loss: 0.0736 - val_accuracy: 0.9770 - val_loss: 0.0786


In [10]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)


281/281 ━━━━━━━━━━━━━━━━━━━━ 69s 244ms/step - accuracy: 0.9765 - loss: 0.0764
Accuracy: 0.9765138030052185


In [11]:
model.save("lstm_fake_news.keras")


joblib.dump(tokenizer, "tokenizer_lstm.pkl")


['tokenizer_lstm.pkl']

In [16]:
def predict_news_lstm(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN)

    pred = model.predict(padded)[0][0]

    if pred > 0.5:
        label = "REAL"
        confidence = pred
    else:
        label = "FAKE"
        confidence = 1 - pred

    return f"{label} ({confidence*100:.1f}% confidence)"

In [17]:
app = gr.Interface(
    fn=predict_news_lstm,

    inputs=gr.Textbox(
        label="Enter news in English",
        placeholder="E.g., Vaccines contain microchips..."
    ),

    outputs=gr.Text(label="Result"),

    title="📰 Fake News Detector",
    description="Enter a news article in English and the AI will tell you if it's REAL or FAKE."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4f02de72523673df62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
